# IMPORTS

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os, sys
import torch
import numpy as np

import CL_inference as cl_inference
N_threads = cl_inference.train_tools.set_N_threads_(N_threads=1)
torch.set_num_threads(N_threads)
torch.set_num_interop_threads(N_threads)
device = cl_inference.train_tools.set_torch_device_()

%load_ext autoreload

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib notebook
plt.style.use('default')
plt.close('all')
font, rcnew = cl_inference.plot_utils.matplotlib_default_config()
mpl.rc('font', **font)
plt.rcParams.update(rcnew)
plt.style.use('tableau-colorblind10')
%config InlineBackend.figure_format = 'retina'

In [ ]:
def dset_normalization(xx, normalize, path_save_norm, path_load_norm, np_name_mean="mean.npy", np_name_std="std.npy"):
    
    if normalize and (path_save_norm is not None) and (path_load_norm is None):
        logging.info('Normalizing data and saving normalization parameters...')
        tmp_xx = np.reshape(xx, tuple([xx.shape[0]*xx.shape[1]*xx.shape[2],] + list(xx.shape[3:])))
        tmp_mean = np.mean(tmp_xx, axis=0)
        tmp_std = np.std(tmp_xx, axis=0)
        if not os.path.exists(path_save_norm):
            os.makedirs(path_save_norm)
        np.save(os.path.join(path_save_norm, np_name_mean), tmp_mean)
        np.save(os.path.join(path_save_norm, np_name_std), tmp_std)
        norm_mean = tmp_mean
        norm_std = tmp_std
        
    elif normalize and (path_load_norm is not None) and (path_save_norm is None):
        logging.info('Loading normalization parameters...')
        norm_mean = np.load(os.path.join(path_load_norm, np_name_mean))
        norm_std = np.load(os.path.join(path_load_norm, np_name_std))
        
    else:
        norm_mean = 0.
        norm_std = 1.
        
    xx = (xx - norm_mean) / norm_std
    
    return xx, norm_mean, norm_std


def func_add_noise_Pk(xx, NN_noise_realizations, kmax, box, factor_kmin_cut, norm_mean, norm_std, gaussian_error_counter_tolerance=10):

    tmp_pk = 10**(xx * norm_std + norm_mean)

    kf = 2.0 * np.pi / box
    kmin = np.log10(factor_kmin_cut * kf)
    N_kk = int((kmax - kmin) / (8 * kf))
    kk_log = np.logspace(kmin, kmax, num=N_kk)
    delta_log10kk = (np.log10(kk_log[1]) - np.log10(kk_log[0])) / 2
    kk_edges_log = 10**np.append(np.log10(kk_log) - delta_log10kk, np.log10(kk_log[-1]) + delta_log10kk)
    delta_kk = np.diff(kk_edges_log)
    
    cosmic_var_gauss_err = np.sqrt((4 * np.pi**2) / (box**3 * kk_log**2 * delta_kk)) * tmp_pk

    valid_sample = False
    while_counter = 0
    while not valid_sample:
        samples_pk = np.random.normal(loc=tmp_pk, scale=cosmic_var_gauss_err, size=(NN_noise_realizations,)+tmp_pk.shape)
        if np.sum(samples_pk < 0) == 0:
            valid_sample = True
        else:
            while_counter += 1
            tmp_indexes = np.where(samples_pk < 0)
            logging.warning(f"WARNING ({while_counter} / {gaussian_error_counter_tolerance}): gaussian error approximation failed. "
                            f"# Corrupted samples = {len(tmp_indexes[0])} / {samples_pk.shape[0] * samples_pk.shape[0]}")
            if while_counter > gaussian_error_counter_tolerance:
                import matplotlib as mpl
                import matplotlib.pyplot as plt
                fig, ax = plt.subplots(1, 1, figsize=(6, 4))
                ax.set_title(f'# Corrupted samples = {len(tmp_indexes[0])} / {samples_pk.shape[0] * samples_pk.shape[0]}', fontsize=16)
                ax.set_xlabel(r'$\mathrm{Wavenumber}\, k \left[ h\, \mathrm{Mpc}^{-1} \right]$')
                ax.set_ylabel(r'$P(k) \left[ \left(h^{-1} \mathrm{Mpc}\right)^{3} \right]$')
                ax.set_xscale('log')
                ax.set_yscale('log')
                ax.plot(kk_log, samples_pk[tmp_indexes[0], tmp_indexes[1]].T, c='k', alpha=0.9, marker=None, lw=0.5, ms=2)
                fig.tight_layout()
                fig.savefig('./gaussian_error_approximation_failed.png')
                assert False, f"ERROR: gaussian error approximation failed!!. Cosmology indexes: {tmp_indexes[0]}. Augmentation indexes: {tmp_indexes[1]}"
    
    xx = (np.log10(samples_pk) - norm_mean) / norm_std
    
    return xx

# Setup

In [ ]:
models_path = "/cosmos_storage/home/dlopez/Projects/CL_inference/models/"

kmax = 0.6 # 0.6, 0.2, -0.2, -0.6, -1.0, -1.4
tmp_CL_str = "Wein" # Wein, VICReg
tmp_dataset_str = "bahamas_illustris"
# all
# v1_v2, v1_v3, v2_v3
# f0_f1, f2_f3, f4_f5, f6_f7, f8_f9
# illustris_eagle, bahamas_illustris, eagle_bahamas

In [ ]:
NN_noise_realizations = 2
model_name_broadest = "Model_vary_all"
save_SBI_path = "/cosmos_storage/home/dlopez/Projects/CL_inference/DATASETS_SBI/kmax_"+str(kmax)

# load config

In [ ]:
tmp_str = "_models_" + tmp_dataset_str + "_kmax_" + str(kmax)
main_name = "only" + "_CL_"                         + tmp_CL_str + tmp_str
models_path = os.path.join(models_path, main_name)
print(models_path)
configs = cl_inference.plot_utils.get_config_files(
    models_path, select_N_best_runs=1,
    wandb_entity="daniellopezcano"
)

sweep_name_0 = list(configs.keys())[0]
config = configs[sweep_name_0]

try:
    print("include_baryon_params:", config['include_baryon_params'])
except:
    config['include_baryon_params'] = False
include_baryon_params = config['include_baryon_params']

try:
    print("dset_type:", config['dset_type'])
except:
    config['dset_type'] = "baccoemu"
dset_type = config['dset_type']

try:
    print("dset_dict:", config['dset_dict'])
except:
    config['dset_dict'] = {
        "add_noise_Pk" : "cosmic_var_gauss",
        "kmax"         : kmax
    }
    try:
        print("box:", config['box'])
        config['dset_dict'].update({'box' : config['box']})
    except:
        config['dset_dict'].update({'box' : 2000})
    try:
        print("factor_kmin_cut:", config['factor_kmin_cut'])
        config['dset_dict'].update({'factor_kmin_cut' : config['factor_kmin_cut']})
    except:
        config['dset_dict'].update({'factor_kmin_cut' : 4})
        
# evalute_mode = 'eval_CL' # "eval_CL", "eval_CL_and_inference", "eval_inference_supervised"
if "only_CL" in main_name:
    evalute_mode = 'eval_CL'
else:
    evalute_mode = 'eval_CL_and_inference'
    
models_encoder, models_inference = cl_inference.evaluation_tools.reload_models(models_path, evalute_mode, configs, device)

if next(models_encoder[list(models_encoder.keys())[0]].parameters()).is_cuda: device = "cuda"
else: device = "cpu"

# Load master normalizer dataset 

In [ ]:
box = config['dset_dict']['box']
factor_kmin_cut = config['dset_dict']['factor_kmin_cut']

In [ ]:
path_load = os.path.join(config['path_load'], "TEST")
model_name = model_name_broadest

np_file_name = "xx.npy"
xx = np.load(os.path.join(path_load, model_name + '_' + np_file_name))

xx_norm, norm_mean, norm_std = dset_normalization(
    xx, normalize=config['normalize'], path_save_norm=None, path_load_norm=os.path.join(config['path_save'], sweep_name_0)
)
xx_norm_noise = func_add_noise_Pk(
    xx_norm, NN_noise_realizations, kmax=kmax, box=box, factor_kmin_cut=factor_kmin_cut, norm_mean=norm_mean, norm_std=norm_std
)
xx_norm_noise_flatten = np.reshape(xx_norm_noise, (np.prod(xx_norm_noise.shape[0:3]), xx_norm_noise.shape[-1]))
xx_norm_noise_flatten_torch = torch.from_numpy(xx_norm_noise_flatten.astype(np.float32)).to(device).contiguous()

hh_noise_flatten_torch = models_encoder[sweep_name_0](xx_norm_noise_flatten_torch)
hh_noise_flatten = hh_noise_flatten_torch.cpu().detach().numpy()
hh_noise = np.reshape(hh_noise_flatten, xx_norm_noise.shape[0:3] + (hh_noise_flatten.shape[-1],))

xx_noise_flatten = (xx_norm_noise_flatten * norm_std + norm_mean)
SBI_mean = np.mean(xx_noise_flatten, axis=0)
SBI_std = np.std(xx_noise_flatten, axis=0)
xx_noise = np.reshape(xx_noise_flatten, xx_norm_noise.shape[0:3] + (xx_noise_flatten.shape[-1],))
xx_save_SBI = (xx_noise - SBI_mean) / SBI_std

SBI_mean_latents = np.mean(hh_noise_flatten, axis=0)
SBI_std_latents = np.std(hh_noise_flatten, axis=0)
hh_save_SBI = (hh_noise - SBI_mean_latents) / SBI_std_latents

In [ ]:
indexes_cosmos = [0, -1]
colors = ["royalblue", "crimson"]
indexes_augs = [0, -1]
lstyls = ["-", "--"]
idx_noise = 0

fig, ax = cl_inference.plot_utils.simple_plot()
for ii, idx_cosmo in enumerate(indexes_cosmos):
    for jj, idx_aug in enumerate(indexes_augs):
        ax.plot(xx[idx_cosmo, idx_aug], c='k', ls=lstyls[jj], lw=5)
        ax.plot(xx_norm_noise[idx_noise, idx_cosmo, idx_aug] * norm_std + norm_mean, c="royalblue", ls=lstyls[jj], lw=3)
        ax.plot(xx_save_SBI[idx_noise, idx_cosmo, idx_aug] * SBI_std + SBI_mean, c="limegreen", ls=lstyls[jj], lw=1)
fig.set_tight_layout(True)

fig, ax = cl_inference.plot_utils.simple_plot()
for ii, idx_cosmo in enumerate(indexes_cosmos):
    for jj, idx_aug in enumerate(indexes_augs):
        ax.plot(xx_norm_noise[idx_noise, idx_cosmo, idx_aug], c="royalblue", ls=lstyls[jj], lw=3)
        ax.plot(xx_save_SBI[idx_noise, idx_cosmo, idx_aug], c="limegreen", ls=lstyls[jj], lw=1)
fig.set_tight_layout(True)

# Save normalized datasets

In [ ]:
for ii, dset_name in enumerate(["TRAIN", "TEST"]):
    path_load = os.path.join(config['path_load'], dset_name)
    for jj, model_name in enumerate(config['list_model_names']+[model_name_broadest]):
        
        print("model_name", model_name)
        
        np_file_name = "xx.npy"
        xx = np.load(os.path.join(path_load, model_name + '_' + np_file_name))

        xx_norm, norm_mean, norm_std = dset_normalization(
            xx, normalize=config['normalize'], path_save_norm=None, path_load_norm=os.path.join(config['path_save'], sweep_name_0)
        )
        xx_norm_noise = func_add_noise_Pk(
            xx_norm, NN_noise_realizations, kmax=kmax, box=box, factor_kmin_cut=factor_kmin_cut, norm_mean=norm_mean, norm_std=norm_std
        )
        xx_norm_noise_flatten = np.reshape(xx_norm_noise, (np.prod(xx_norm_noise.shape[0:3]), xx_norm_noise.shape[-1]))
        xx_norm_noise_flatten_torch = torch.from_numpy(xx_norm_noise_flatten.astype(np.float32)).to(device).contiguous()

        hh_noise_flatten_torch = models_encoder[sweep_name_0](xx_norm_noise_flatten_torch)
        hh_noise_flatten = hh_noise_flatten_torch.cpu().detach().numpy()
        hh_noise = np.reshape(hh_noise_flatten, xx_norm_noise.shape[0:3] + (hh_noise_flatten.shape[-1],))
        
        xx_noise_flatten = (xx_norm_noise_flatten * norm_std + norm_mean)
        xx_noise = np.reshape(xx_noise_flatten, xx_norm_noise.shape[0:3] + (xx_noise_flatten.shape[-1],))
        xx_save_SBI = (xx_noise - SBI_mean) / SBI_std

        hh_save_SBI = (hh_noise - SBI_mean_latents) / SBI_std_latents
        
        np_file_name = "cosmos.npy"
        theta = np.load(os.path.join(path_load, model_name + '_' + np_file_name))
        np_file_name = "aug_params.npy"
        aug_params = np.load(os.path.join(path_load, model_name + '_' + np_file_name))
        np_file_name = "extended_aug_params.npy"
        ext_aug_params = np.load(os.path.join(path_load, model_name + '_' + np_file_name))        

        tmp_save_str = os.path.join(save_SBI_path, dset_name, "Pk", model_name)
        if not os.path.exists(tmp_save_str):
            os.makedirs(tmp_save_str)
        np.save(os.path.join(tmp_save_str, "theta.npy"), theta)
        np.save(os.path.join(tmp_save_str, "ext_aug_params.npy"), ext_aug_params)
        np.save(os.path.join(tmp_save_str, "xx.npy"), xx_save_SBI)
        
        tmp_save_str = os.path.join(save_SBI_path, dset_name, "latents_"+tmp_CL_str, tmp_dataset_str, model_name)
        if not os.path.exists(tmp_save_str):
            os.makedirs(tmp_save_str)
        np.save(os.path.join(tmp_save_str, "theta.npy"), theta)
        np.save(os.path.join(tmp_save_str, "ext_aug_params.npy"), ext_aug_params)
        np.save(os.path.join(tmp_save_str, "xx.npy"), hh_save_SBI)
        
#         fig, ax = cl_inference.plot_utils.simple_plot()
#         for ii, idx_cosmo in enumerate(indexes_cosmos):
#             for jj, idx_aug in enumerate(indexes_augs):
#                 ax.plot(xx[idx_cosmo, idx_aug], c='k', ls=lstyls[jj], lw=5)
#                 ax.plot(xx_norm_noise[idx_noise, idx_cosmo, idx_aug] * norm_std + norm_mean, c="royalblue", ls=lstyls[jj], lw=3)
#                 ax.plot(xx_save_SBI[idx_noise, idx_cosmo, idx_aug] * SBI_std + SBI_mean, c="limegreen", ls=lstyls[jj], lw=1)
#         fig.set_tight_layout(True)

#         fig, ax = cl_inference.plot_utils.simple_plot()
#         for ii, idx_cosmo in enumerate(indexes_cosmos):
#             for jj, idx_aug in enumerate(indexes_augs):
#                 ax.plot(xx_norm_noise[idx_noise, idx_cosmo, idx_aug], c="royalblue", ls=lstyls[jj], lw=3)
#                 ax.plot(xx_save_SBI[idx_noise, idx_cosmo, idx_aug], c="limegreen", ls=lstyls[jj], lw=1)
#         fig.set_tight_layout(True)